# Studi Kasus 5: Profil Ketenagakerjaan

Notebook ini mengelompokkan UMKM berdasarkan kesejahteraan pekerja, rasio gender, usia pekerja, dan asuransi BPJS.

**Metode**: K-Means Clustering + Cuckoo Search Algorithm Optimization


In [ ]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath("../src"))
from preprocessing import preprocess_ketenagakerjaan
from csa_core import hitung_optimal_k_elbow, cuckoo_search_kmeans, final_kmeans, plot_hasil_cluster, evaluasi_kualitas_klasterisasi

# 1. Preprocessing Data Khusus Skenario
df_raw, df_scaled, list_fitur = preprocess_ketenagakerjaan("../data/Data Set UMKM.xlsx")
X_scaled = df_scaled.values

print(f"Data shape: {X_scaled.shape}")
print(f"Fitur: {list_fitur}")
df_raw.head()


In [ ]:
# 2. Mencari K-Optimal secara Otomatis
print("\nMencari jumlah klaster (K) optimal...")
optimal_k = hitung_optimal_k_elbow(X_scaled, max_k=8)


In [ ]:
# 3. Proses Training (Fitting) dengan Cuckoo Search Algorithm
print(f"\nMemulai Cuckoo Search dengan K={optimal_k}...")
best_nest, fitness_history = cuckoo_search_kmeans(
    X=X_scaled, 
    k=optimal_k, 
    n_nests=10, 
    max_iter=30, 
    pa=0.25
)

# 4. Fine-Tuning dengan K-Means konvensional
print("\nMelakukan Fine-Tuning K-Means...")
labels, final_centroids = final_kmeans(X_scaled, best_nest)


In [ ]:
# 5. Evaluasi Metrik & Visualisasi
evaluasi_kualitas_klasterisasi(X_scaled, labels, final_centroids)

# Memasukkan hasil klaster ke dataframe asli
df_raw["Cluster"] = labels
print("\nDistribusi Klaster:")
print(df_raw["Cluster"].value_counts())

# Plot hasil
plot_hasil_cluster(X_scaled, final_centroids, labels, list_fitur)
